In [ ]:
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pandas as pd

def parquet_summary(path, time_col="created_at", sample_rows=5):
    print(f"\n================ {path} ================")
    
    # Open file via metadata only
    pf = pq.ParquetFile(path)
    
    # Two types of schema:
    schema_parquet = pf.schema
    schema_arrow = pf.schema_arrow
    
    print("\nSchema from metadata:")
    print(schema_parquet)
    
    # Column names & logical types
    print("\nColumns & types:")
    for field in schema_arrow:
        print(f" - {field.name:20s} : {field.type}")
    
    # Fast row count from metadata
    n_rows = pf.metadata.num_rows
    print(f"\nRow count (from metadata): {n_rows:,}")
    
    # Small sample without loading everything
    try:
        first_batch = next(pf.iter_batches(batch_size=sample_rows))
        df_sample = first_batch.to_pandas()
        print(f"\nHead (first {len(df_sample)} rows):")
        print(df_sample)
    except StopIteration:
        print("\nFile appears to be empty.")
    
    # Time range for the given time column (if present)
    if time_col is not None and time_col in schema_arrow.names:
        mins = []
        maxs = []
        for rg_idx in range(pf.num_row_groups):
            # Read only that column for each row group
            tbl = pf.read_row_group(rg_idx, columns=[time_col])
            col = tbl[time_col]  # this is a ChunkedArray
            
            if col.null_count == len(col):
                continue  # all null in this row group
            
            # Use pyarrow.compute instead of col.min() / col.max()
            mins.append(pc.min(col).as_py())
            maxs.append(pc.max(col).as_py())
        
        if mins:
            print(f"\nTime range for '{time_col}':")
            print(f" - min: {min(mins)}")
            print(f" - max: {max(maxs)}")
        else:
            print(f"\nTime range for '{time_col}': all values are null")
    elif time_col is not None:
        print(f"\nTime range: column '{time_col}' not found.")


In [ ]:
parquet_summary("chunk_0_follow.parquet", time_col="created_at")


================ chunk_0_follow.parquet ================

Schema from metadata:
required group field_id=-1 schema {
  optional int64 field_id=-1 did_id;
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 subject_id;
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - subject_id           : int64

Row count (from metadata): 5,000,000

Head (first 5 rows):
   did_id           rkey                       created_at  subject_id
0       1  3laxsqgevg324 2024-11-15 07:04:15.472000+00:00    32919732
1       1  3laykpiwyfy25 2024-11-15 14:13:14.331000+00:00    10541982
2       1  3lbsl5ra5zc23 2024-11-25 22:30:25.945000+00:00    21894245
3       1  3lbsl66ze4s2v 2024-11-25 22:30:40.392000+00:00    26881587
4       1  3lbtgjh5suq2z

In [38]:
parquet_summary("chunk_0_likes.parquet", time_col="created_at")


================ chunk_0_likes.parquet ================

Schema from metadata:
required group field_id=-1 schema {
  optional int64 field_id=-1 did_id;
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional group field_id=-1 subject {
    optional int64 field_id=-1 did_id;
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - subject              : struct<did_id: int64, collection: string, rkey: string>

Row count (from metadata): 20,000,000

Head (first 5 rows):
   did_id           rkey                       created_at  \
0       3  3lb2zabhrp227 2024-11-16 13:38:28.769000+00:00   
1       3  3lb2zi4jf3s27 2024-11-16 13:42:52.012000+00:00   


In [39]:
parquet_summary("chunk_0_posts.parquet", time_col="created_at")


================ chunk_0_posts.parquet ================

Schema from metadata:
required group field_id=-1 schema {
  optional int64 field_id=-1 did_id;
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 languages (JSON);
  optional binary field_id=-1 labels (JSON);
  optional binary field_id=-1 tags (JSON);
  optional binary field_id=-1 embed_type (String);
  optional group field_id=-1 embed_record {
    optional int64 field_id=-1 did_id;
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
  optional binary field_id=-1 embed_external_uri (String);
  optional binary field_id=-1 embed_images (JSON);
  optional binary field_id=-1 embed_media (JSON);
  optional binary field_id=-1 embed_video (JSON);
  optional group field_id=-1 reply {
    optional group fi

In [40]:
parquet_summary("chunk_0_reposts.parquet", time_col="created_at")


================ chunk_0_reposts.parquet ================

Schema from metadata:
required group field_id=-1 schema {
  optional int64 field_id=-1 did_id;
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional group field_id=-1 subject {
    optional int64 field_id=-1 did_id;
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - subject              : struct<did_id: int64, collection: string, rkey: string>

Row count (from metadata): 5,000,000

Head (first 5 rows):
   did_id           rkey                       created_at  \
0       3  3lg7gwjalgh22 2025-01-20 23:00:10.850000+00:00   
1       3  3lgtavbqzip2c 2025-01-28 20:05:21.769000+00:00   

In [41]:
parquet_summary("lists.parquet", time_col="created_at")


================ lists.parquet ================

Schema from metadata:
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 name (String);
  optional binary field_id=-1 description (String);
  optional group field_id=-1 avatar {
    optional group field_id=-1 ref {
      optional binary field_id=-1 / (String);
    }
    optional int64 field_id=-1 size (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 mimeType (String);
  }
  optional binary field_id=-1 purpose (String);
  optional binary field_id=-1 labels (JSON);
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - name                 : string

In [42]:
parquet_summary("list_blocks.parquet", time_col="created_at")


================ list_blocks.parquet ================

Schema from metadata:
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional group field_id=-1 subject {
    optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - subject              : struct<did_id: int64, collection: string, rkey: string>

Row count (from metadata): 4,285,907

Head (first 5 rows):
   did_id           rkey                       created_at  \
0      44  3l3e5iyzwxg2l 2024-09-04 20:17:44.66400

In [43]:
parquet_summary("list_items.parquet", time_col="created_at")


================ list_items.parquet ================

Schema from metadata:
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional group field_id=-1 list {
    optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 collection (String);
    optional binary field_id=-1 rkey (String);
  }
  optional int64 field_id=-1 subject_id (Int(bitWidth=64, isSigned=true));
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - list                 : struct<did_id: int64, collection: string, rkey: string>
 - subject_id           : int64

Row count (from metadata): 43,543,360

Head (first 5 rows):
   did_


Time range for 'created_at':
 - min: 0010-01-01 12:16:35.797000+00:00
 - max: 2025-04-16 06:47:04.294000+00:00


In [44]:
parquet_summary("profiles.parquet", time_col="created_at")


================ profiles.parquet ================

Schema from metadata:
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 description (String);
  optional binary field_id=-1 labels (JSON);
  optional group field_id=-1 avatar {
    optional group field_id=-1 ref {
      optional binary field_id=-1 / (String);
    }
    optional int64 field_id=-1 size (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 mimeType (String);
  }
  optional group field_id=-1 banner {
    optional group field_id=-1 ref {
      optional binary field_id=-1 / (String);
    }
    optional int64 field_id=-1 size (Int(bitWidth=64, isSigned=true));
    optional binary field_id=-1 mimeType (String);
  }
  o

In [45]:
parquet_summary("blocks.parquet", time_col="created_at")


================ blocks.parquet ================

Schema from metadata:
required group field_id=-1 duckdb_schema {
  optional int64 field_id=-1 did_id (Int(bitWidth=64, isSigned=true));
  optional binary field_id=-1 rkey (String);
  optional int64 field_id=-1 created_at (Timestamp(isAdjustedToUTC=true, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 subject_id (Int(bitWidth=64, isSigned=true));
}


Columns & types:
 - did_id               : int64
 - rkey                 : string
 - created_at           : timestamp[us, tz=UTC]
 - subject_id           : int64

Row count (from metadata): 120,088,510

Head (first 5 rows):
   did_id           rkey                       created_at  subject_id
0      16  3llypftynss2z 2025-04-04 15:14:26.546000+00:00    13680162
1      28  3kky4egdjro2s 2024-02-09 10:42:14.189000+00:00    15709977
2      28  3klcfg4dy3r2n 2024-02-13 12:50:51.322000+00:00    28594525
3      28  3km5e2rejo726 

In [46]:
def sample_parquet(path, n=5000):
    """
    Read a small random sample from a huge parquet file using row groups.
    """
    pf = pq.ParquetFile(path)

    samples = []
    for rg in range(pf.num_row_groups):
        batch = pf.read_row_group(rg)
        df = batch.to_pandas()

        # Randomly sample a few rows per row group
        samples.append(df.sample(min(n // pf.num_row_groups, len(df))))

        if len(samples) * (n // pf.num_row_groups) >= n:
            break

    return pa.Table.from_pandas(
        pandas.concat(samples, ignore_index=True)
    )


1. Compare LABELS across LISTS / PROFILES / POSTS

In [ ]:
import pandas as pd
import json
from ast import literal_eval
from collections import Counter


def extract_label_values(raw):
    """
    raw = a stringified dict coming from parquet
    returns list of 'val' fields inside: dict['values']
    """
    if pd.isna(raw) or raw is None:
        return []

    # Handle JSON-like but not valid JSON
    try:
        obj = literal_eval(raw)
    except Exception:
        try:
            obj = json.loads(raw)
        except Exception:
            return []

    if not isinstance(obj, dict):
        return []

    vals = obj.get("values", [])
    out = []
    for v in vals:
        if isinstance(v, dict) and "val" in v:
            out.append(str(v["val"]))
    return out


# --------------------------------------------------
# LOAD PARQUETS
# --------------------------------------------------

lists = pd.read_parquet("lists.parquet", columns=["labels"])
profiles = pd.read_parquet("profiles.parquet", columns=["labels"])
posts = pd.read_parquet("chunk_0_posts.parquet", columns=["labels"])

# --------------------------------------------------
# EXTRACT ALL LABEL VALUES
# --------------------------------------------------

def get_label_counter(df):
    counter = Counter()
    for raw in df["labels"].dropna():
        labels = extract_label_values(raw)
        counter.update(labels)
    return counter


lists_counter = get_label_counter(lists)
profiles_counter = get_label_counter(profiles)
posts_counter = get_label_counter(posts)

# --------------------------------------------------
# PRINT COUNTS
# --------------------------------------------------

print("\n=== LABEL COUNTS IN lists.parquet ===")
print(pd.DataFrame(lists_counter.most_common(), columns=["label", "count"]))

print("\n=== LABEL COUNTS IN profiles.parquet ===")
print(pd.DataFrame(profiles_counter.most_common(), columns=["label", "count"]))

print("\n=== LABEL COUNTS IN posts.parquet ===")
print(pd.DataFrame(posts_counter.most_common(), columns=["label", "count"]))



=== LABEL COUNTS IN lists.parquet ===
                             label  count
0               app.together.group      3
1  app.bsky.graph.defs/starterpack      1
2                     starterpacks      1

=== LABEL COUNTS IN profiles.parquet ===
                                  label    count
0                   !no-unauthenticated  1579998
1   bridged-from-bridgy-fed-activitypub     1039
2           bridged-from-bridgy-fed-web      120
3                                 !hide        2
4                                    少女        1
5                              curation        1
6                                     🦀        1
7                                  Test        1
8                               GameDev        1
9                                  ぬるぬる        1
10                                 neko        1
11                         Power Ranger        1
12                                 test        1
13                          三 🦀 三 🦀 三 🦀        1
14              

2. Compare LABELS across LANGUAGES in POSTS

In [50]:
def labels_by_language(posts_path, sample_size=5000):
    pf = pq.ParquetFile(posts_path)
    lang_to_labels = {}

    for rg in range(pf.num_row_groups):
        batch = pf.read_row_group(rg, columns=["languages", "labels"])

        langs = batch["languages"].to_pylist()
        labels = batch["labels"].to_pylist()

        for lang_raw, label_raw in zip(langs, labels):
            if lang_raw is None:
                continue

            # parse language JSON
            try:
                lang_list = json.loads(lang_raw)
                lang = lang_list[0] if isinstance(lang_list, list) else str(lang_raw)
            except:
                lang = str(lang_raw)

            # parse labels JSON
            label_set = set()
            if label_raw:
                try:
                    label_list = json.loads(label_raw)
                    if isinstance(label_list, list):
                        label_set.update(label_list)
                except:
                    label_set.add(str(label_raw))

            lang_to_labels.setdefault(lang, set()).update(label_set)

        if sum(len(v) for v in lang_to_labels.values()) > sample_size:
            break

    return lang_to_labels


lang_labels = labels_by_language("chunk_0_posts.parquet")

for lang, labs in lang_labels.items():
    print(f"Language: {lang} → Labels: {labs}")


Language: sv → Labels: set()
Language: pt → Labels: set()
Language: en → Labels: set()
Language: de → Labels: set()
Language: ja → Labels: set()
Language: pl → Labels: set()
Language: ko → Labels: set()
Language: th → Labels: set()
Language: es → Labels: set()
Language: fr → Labels: set()
Language: zh → Labels: set()
Language: Unknown → Labels: set()
Language: fa → Labels: set()
Language: it → Labels: set()
Language: fi → Labels: set()
Language: in → Labels: set()
Language: nl → Labels: set()
Language: ar → Labels: set()
Language: id → Labels: set()
Language: tr → Labels: set()
Language: ru → Labels: set()
Language: uk → Labels: set()
Language: hu → Labels: set()
Language: et → Labels: set()
Language: ab → Labels: set()
Language: [] → Labels: set()
Language: vi → Labels: set()
Language: ca → Labels: set()
Language: bg → Labels: set()
Language: gl → Labels: set()
Language: el → Labels: set()
Language: da → Labels: set()
Language: nb → Labels: set()
Language: cs → Labels: set()
Language:

3. Check reply structure: only root+parent or more?

In [54]:
def inspect_replies(path, sample_size=2000):
    pf = pq.ParquetFile(path)
    depths = Counter()

    for rg in range(pf.num_row_groups):
        try:
            replies = pf.read_row_group(rg, columns=["reply"])["reply"].to_pylist()
        except KeyError:
            print("No reply column found.")
            return

        for r in replies:
            if r is None:
                continue

            try:
                r = r if isinstance(r, dict) else json.loads(r)
            except:
                continue

            # depth = number of nested dictionaries
            def depth_of(d):
                if not isinstance(d, dict):
                    return 0
                return 1 + max([depth_of(v) for v in d.values()] or [0])

            depths[depth_of(r)] += 1

            if sum(depths.values()) >= sample_size:
                break

    print("Reply structure depth distribution:")
    for d, count in sorted(depths.items()):
        print(f"Depth {d}: {count} samples")

    if max(depths) <= 2:
        print("\n Replies contain ONLY root + parent → No thread history.")
    else:
        print("\n Some replies contain deeper content → longer chains exist.")

inspect_replies("chunk_0_posts.parquet", sample_size=2000)


Reply structure depth distribution:
Depth 2: 2004 samples

 Replies contain ONLY root + parent → No thread history.
